# Original-Dataset MSE: DINOv2 vs. V-JEPA2

First results plot for the two encoders trained on the **original (pre-enrichment) dataset**: final test MSE, bootstrap error bars, and the split-half noise floor as a reference ceiling.

**V-JEPA2 hasn't been trained yet** — its bar is a placeholder at 0 so the x-axis is already in place; swap in the real CSV path once it's ready (see the config cell).

**Error bars** = bootstrap uncertainty: resample the test videos with replacement (all 12 categories travel together per video), recompute MSE each time, take the mean and 95% percentile interval across repetitions. This reflects how much the MSE estimate would wobble with a different sample of test videos — not retraining variance (each model was trained once).

**Noise floor** = split-half reliability of the human ground truth itself (MSE between two independent groups of raters) — the best score any model could possibly achieve, since it reflects the irreducible noise in the human data.

**Caveat on the noise floor** (read before trusting it): it's pulled from the existing cached `Reliability_all_duration.csv` (the same one used throughout `evaluation_plots_clean.ipynb`), averaged across its duration bins at the nearest available sample size to 32. That cache reflects the *main* test set's reliability curve — it is **not** filtered to the exact 602 original-dataset videos DINOv2 was evaluated on here, so treat it as a good approximation, not an exact match. A stricter version (computed only from the original-dataset test videos) would need `reliability.compute_reliability_scaling(..., grouping_variable=None)` run directly on the raw human trial data restricted to those video IDs — noted as a follow-up in the last cell, not implemented here since it's a much heavier computation and this is meant to be a fast first plot.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Config

In [ ]:
REPO_ROOT = "/braintree/home/aicha/master_thesis"
sys.path.insert(0, REPO_ROOT)

DINOV2_PREDS_CSV = (
    f"{REPO_ROOT}/result_csvs/"
    "dino_v2-dino_v2_60frames_15epochs_lr_0.001_wd_0.8_baseline_fps_4_preds_60Frames_AttentionalPooling_og_dataset.csv"
)
VJEPA2_PREDS_CSV = None  # not trained yet -- set this once results exist, e.g.:
# VJEPA2_PREDS_CSV = f"{REPO_ROOT}/result_csvs/vjepa2-...og_dataset.csv"

MODEL_LABELS = ["DINOv2 Large", "V-JEPA2"]

N_BOOTSTRAP = 100
BOOTSTRAP_SEED = 42

NOISE_FLOOR_SAMPLE_SIZE = 32

## Load predictions and compute bootstrap MSE

In [ ]:
def load_predictions(path):
    df = pd.read_csv(path)
    if "video_type" in df.columns:
        df = df.loc[df["video_type"] == "test"]
    return df


def bootstrap_mse(df, n_reps=N_BOOTSTRAP, seed=BOOTSTRAP_SEED):
    """Resample test videos with replacement (all categories travel together per
    video), recompute MSE each rep. Returns (mean, ci_low, ci_high)."""
    rng = np.random.default_rng(seed)
    grouped = {vid: g[["prediction", "labels"]].to_numpy() for vid, g in df.groupby("videoID")}
    video_ids = np.array(list(grouped.keys()))

    mses = np.empty(n_reps)
    for i in range(n_reps):
        sampled_ids = rng.choice(video_ids, size=len(video_ids), replace=True)
        stacked = np.concatenate([grouped[v] for v in sampled_ids], axis=0)
        preds, labels = stacked[:, 0], stacked[:, 1]
        mses[i] = np.mean((preds - labels) ** 2)

    return mses.mean(), np.percentile(mses, 2.5), np.percentile(mses, 97.5)


results = {}

dinov2_df = load_predictions(DINOV2_PREDS_CSV)
print(f"DINOv2: {dinov2_df['videoID'].nunique()} videos, {len(dinov2_df)} rows")
results["DINOv2 Large"] = bootstrap_mse(dinov2_df)

if VJEPA2_PREDS_CSV is not None:
    vjepa2_df = load_predictions(VJEPA2_PREDS_CSV)
    print(f"V-JEPA2: {vjepa2_df['videoID'].nunique()} videos, {len(vjepa2_df)} rows")
    results["V-JEPA2"] = bootstrap_mse(vjepa2_df)
else:
    print("V-JEPA2: no results yet -- using placeholder value 0")
    results["V-JEPA2"] = (0.0, 0.0, 0.0)

results

## Noise floor (approximate — see caveat above)

In [ ]:
from reliability import load_reliability_durations

reliability_df = load_reliability_durations()

nf = reliability_df.loc[
    (reliability_df["metric"] == "MSE")
    & (reliability_df["type"] == "prediction")
    & (reliability_df["category"] == "all")
    & (reliability_df["kind"] == "all")
]

nearest_sample_size = nf["Sample size"].iloc[(nf["Sample size"] - NOISE_FLOOR_SAMPLE_SIZE).abs().argsort().iloc[0]]
print(f"Using nearest available Sample size: {nearest_sample_size}")

nf_at_n = nf.loc[nf["Sample size"] == nearest_sample_size]

# average across duration bins per repetition, then take the mean + 95% percentile
# interval across repetitions -- consistent with the bootstrap error bars above
per_rep = nf_at_n.groupby("Repetition")["Reliability"].mean()
noise_floor_mean = per_rep.mean()
noise_floor_low = np.percentile(per_rep, 2.5)
noise_floor_high = np.percentile(per_rep, 97.5)

print(f"Noise floor (MSE): {noise_floor_mean:.4f}  [{noise_floor_low:.4f}, {noise_floor_high:.4f}]")

## Plot

In [ ]:
DARK = "#413C3A"
TEAL = "#00A79F"
PLACEHOLDER_GRAY = "#CAC7C7"

fig, ax = plt.subplots(figsize=(6, 5))

x = np.arange(len(MODEL_LABELS))
means = [results[label][0] for label in MODEL_LABELS]
err_low = [results[label][0] - results[label][1] for label in MODEL_LABELS]
err_high = [results[label][2] - results[label][0] for label in MODEL_LABELS]

colors = [TEAL if VJEPA2_PREDS_CSV is not None or label != "V-JEPA2" else PLACEHOLDER_GRAY for label in MODEL_LABELS]

bars = ax.bar(x, means, width=0.5, color=colors, edgecolor=DARK, linewidth=1.0)
ax.errorbar(x, means, yerr=[err_low, err_high], fmt="none", ecolor=DARK, capsize=5, linewidth=1.5)

for xi, m in zip(x, means):
    ax.text(xi, m + max(means + [noise_floor_mean]) * 0.03, f"{m:.4f}", ha="center", fontsize=10, color=DARK)

if VJEPA2_PREDS_CSV is None:
    ax.text(1, max(means + [noise_floor_mean]) * 0.15, "pending", ha="center", fontsize=10, style="italic", color=DARK)

ax.axhline(noise_floor_mean, color="black", linestyle="--", linewidth=1.2, label="Noise floor")
ax.axhspan(noise_floor_low, noise_floor_high, color="black", alpha=0.08)

ax.set_xticks(x)
ax.set_xticklabels(MODEL_LABELS)
ax.set_ylabel("Test MSE")
ax.set_title("MSE on the original dataset")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", alpha=0.3)

fig.tight_layout()
fig.savefig("original_dataset_mse_plot.png", dpi=150)
plt.show()

## Follow-up (not implemented here)

For a noise floor computed *exactly* on the original-dataset test videos (rather than the approximation above): pull the `videoID`s from `dinov2_df`, filter `data_loaders.load_humanTrialData()`'s output to matching `File_ID`s, and call `reliability.compute_reliability_scaling(df_selected, grouping_variable=None, level='all')` directly, then `apply_reliability_prediction(...)` to extrapolate to n=32. Heavier (raw trial-level data + refitting the reliability-vs-sample-size curve), so left as a follow-up rather than blocking this first plot.